# Chemical Formula Parser — molar mass from a chemical formula

This notebook comes from my **CSE 111** "student-chosen" work at BYU-Idaho: parse a chemical
formula like `H2O` or `C6H12O6` into its element counts, then compute the molar mass from a
periodic-table lookup.

**Honest note on how this runs.** In my original assignment the pieces lived in *two separate
files* — a periodic-table dictionary module and a formula-parser module — and the folder they
ended up in had them split apart (the parser file tried to `from formula import parse_formula`,
but no `formula.py` sat next to it). Neither ran standalone as checked out. For this notebook I
**re-paired my own two files into one place** so the imports resolve and it executes
top-to-bottom. The logic below is entirely mine; the only change is putting the matched halves
back together and dropping the broken cross-file import.

The flow is: **periodic table (dict) -> parser (raises a custom exception) -> molar-mass
computation -> a bad-formula demo -> inline `assert` checks.**

## 1. The periodic table as a dict of lists

Each **key** is an element symbol; each **value** is a `[name, atomic_mass]` list. Keying on
the symbol makes `periodic_table["O"]` an instant lookup, and the `ATOMIC_MASS_INDEX` constant
keeps the inner-list access readable.

In [1]:
def make_periodic_table():
    """Return a dict mapping element symbol -> [name, atomic_mass]."""
    periodic_table_dict = {
        "Ac": ["Actinium", 227], "Ag": ["Silver", 107.8682],
        "Al": ["Aluminum", 26.9815386], "Ar": ["Argon", 39.948],
        "As": ["Arsenic", 74.9216], "At": ["Astatine", 210],
        "Au": ["Gold", 196.966569], "B": ["Boron", 10.811],
        "Ba": ["Barium", 137.327], "Be": ["Beryllium", 9.012182],
        "Bi": ["Bismuth", 208.9804], "Br": ["Bromine", 79.904],
        "C": ["Carbon", 12.0107], "Ca": ["Calcium", 40.078],
        "Cd": ["Cadmium", 112.411], "Ce": ["Cerium", 140.116],
        "Cl": ["Chlorine", 35.453], "Co": ["Cobalt", 58.933195],
        "Cr": ["Chromium", 51.9961], "Cs": ["Cesium", 132.9054519],
        "Cu": ["Copper", 63.546], "Dy": ["Dysprosium", 162.5],
        "Er": ["Erbium", 167.259], "Eu": ["Europium", 151.964],
        "F": ["Fluorine", 18.9984032], "Fe": ["Iron", 55.845],
        "Fr": ["Francium", 223], "Ga": ["Gallium", 69.723],
        "Gd": ["Gadolinium", 157.25], "Ge": ["Germanium", 72.64],
        "H": ["Hydrogen", 1.00794], "He": ["Helium", 4.002602],
        "Hf": ["Hafnium", 178.49], "Hg": ["Mercury", 200.59],
        "Ho": ["Holmium", 164.93032], "I": ["Iodine", 126.90447],
        "In": ["Indium", 114.818], "Ir": ["Iridium", 192.217],
        "K": ["Potassium", 39.0983], "Kr": ["Krypton", 83.798],
        "La": ["Lanthanum", 138.90547], "Li": ["Lithium", 6.941],
        "Lu": ["Lutetium", 174.9668], "Mg": ["Magnesium", 24.305],
        "Mn": ["Manganese", 54.938045], "Mo": ["Molybdenum", 95.96],
        "N": ["Nitrogen", 14.0067], "Na": ["Sodium", 22.98976928],
        "Nb": ["Niobium", 92.90638], "Nd": ["Neodymium", 144.242],
        "Ne": ["Neon", 20.1797], "Ni": ["Nickel", 58.6934],
        "Np": ["Neptunium", 237], "O": ["Oxygen", 15.9994],
        "Os": ["Osmium", 190.23], "P": ["Phosphorus", 30.973762],
        "Pa": ["Protactinium", 231.03588], "Pb": ["Lead", 207.2],
        "Pd": ["Palladium", 106.42], "Pm": ["Promethium", 145],
        "Po": ["Polonium", 209], "Pr": ["Praseodymium", 140.90765],
        "Pt": ["Platinum", 195.084], "Pu": ["Plutonium", 244],
        "Ra": ["Radium", 226], "Rb": ["Rubidium", 85.4678],
        "Re": ["Rhenium", 186.207], "Rh": ["Rhodium", 102.9055],
        "Rn": ["Radon", 222], "Ru": ["Ruthenium", 101.07],
        "S": ["Sulfur", 32.065], "Sb": ["Antimony", 121.76],
        "Sc": ["Scandium", 44.955912], "Se": ["Selenium", 78.96],
        "Si": ["Silicon", 28.0855], "Sm": ["Samarium", 150.36],
        "Sn": ["Tin", 118.71], "Sr": ["Strontium", 87.62],
        "Ta": ["Tantalum", 180.94788], "Tb": ["Terbium", 158.92535],
        "Tc": ["Technetium", 98], "Te": ["Tellurium", 127.6],
        "Th": ["Thorium", 232.03806], "Ti": ["Titanium", 47.867],
        "Tl": ["Thallium", 204.3833], "Tm": ["Thulium", 168.93421],
        "U": ["Uranium", 238.02891], "V": ["Vanadium", 50.9415],
        "W": ["Tungsten", 183.84], "Xe": ["Xenon", 131.293],
        "Y": ["Yttrium", 88.90585], "Yb": ["Ytterbium", 173.054],
        "Zn": ["Zinc", 65.38], "Zr": ["Zirconium", 91.224],
    }
    return periodic_table_dict


# Named indexes into each element's [name, atomic_mass] list.
NAME_INDEX = 0
ATOMIC_MASS_INDEX = 1

periodic_table = make_periodic_table()
print(f"Loaded {len(periodic_table)} elements.")
print("Oxygen ->", periodic_table["O"])

Loaded 94 elements.
Oxygen -> ['Oxygen', 15.9994]


## 2. The parser and its custom exception

`parse_formula` walks the formula string with a small recursive helper (`parse_r`) so it can
handle parenthesized groups like `(CH2)12`. When it hits something invalid — an unknown symbol,
an unmatched parenthesis, a zero quantity — it raises my own **`FormulaError`**, which
subclasses `ValueError`. Defining that one-line exception class is the whole reason a caller can
catch a *formula* problem specifically.

In [2]:
class FormulaError(ValueError):
    """Raised by parse_formula when a chemical formula is invalid.

    Subclasses ValueError, so callers can catch it precisely (except FormulaError)
    or broadly (except ValueError).
    """


def parse_formula(formula, periodic_table_dict):
    """Convert a chemical formula string into a list of [symbol, quantity] pairs.

    "H2O"             -> [["H", 2], ["O", 1]]
    "PO4H2(CH2)12CH3" -> [["P", 1], ["O", 4], ["H", 29], ["C", 13]]
    """
    assert isinstance(formula, str), "formula must be a string"
    assert isinstance(periodic_table_dict, dict), "periodic_table_dict must be a dict"

    def parse_quant(formula, index):
        quant = 1
        if index < len(formula) and formula[index].isdecimal():
            if formula[index] == "0":
                raise FormulaError("invalid formula, quantity begins with zero (0); "
                    "perhaps you meant capital O for Oxygen", formula, index)
            start = index
            index += 1
            while index < len(formula) and formula[index].isdecimal():
                index += 1
            quant = int(formula[start:index])
        return quant, index

    def get_quant(elem_dict, symbol):
        return 0 if symbol not in elem_dict else elem_dict[symbol]

    def parse_r(formula, index, level):
        start_index = index
        start_level = level
        elem_dict = {}
        while index < len(formula):
            ch = formula[index]
            if ch == "(":
                group_dict, index = parse_r(formula, index + 1, level + 1)
                quant, index = parse_quant(formula, index)
                for symbol in group_dict:
                    prev = get_quant(elem_dict, symbol)
                    elem_dict[symbol] = prev + group_dict[symbol] * quant
            elif ch.isalpha():
                symbol = formula[index:index + 2]
                if symbol in periodic_table_dict:
                    index += 2
                else:
                    symbol = formula[index:index + 1]
                    if symbol in periodic_table_dict:
                        index += 1
                    else:
                        raise FormulaError(
                            f"invalid formula; unknown element symbol: {symbol}",
                            formula, index)
                quant, index = parse_quant(formula, index)
                prev = get_quant(elem_dict, symbol)
                elem_dict[symbol] = prev + quant
            elif ch == ")":
                if level == 0:
                    raise FormulaError("invalid formula; unmatched close parenthesis",
                        formula, index)
                level -= 1
                index += 1
                break
            else:
                if ch.isdecimal():
                    message = "invalid formula"
                else:
                    message = f"invalid formula; illegal character: {ch}"
                raise FormulaError(message, formula, index)
        if level > 0 and level >= start_level:
            raise FormulaError("invalid formula; unmatched open parenthesis",
                formula, start_index - 1)
        return elem_dict, index

    elem_dict, _ = parse_r(formula, 0, 0)
    return list(elem_dict.items())


# Quick look at what the parser returns.
print("H2O             ->", parse_formula("H2O", periodic_table))
print("C6H12O6         ->", parse_formula("C6H12O6", periodic_table))
print("PO4H2(CH2)12CH3 ->", parse_formula("PO4H2(CH2)12CH3", periodic_table))

H2O             -> [('H', 2), ('O', 1)]
C6H12O6         -> [('C', 6), ('H', 12), ('O', 6)]
PO4H2(CH2)12CH3 -> [('P', 1), ('O', 4), ('H', 29), ('C', 13)]


## 3. Molar mass from the parsed formula

`compute_molar_mass` consumes the `[symbol, quantity]` list from the parser and looks each
atomic mass up in the periodic-table dict. Water (`H2O`) comes out to ~18.015 g/mol and glucose
(`C6H12O6`) to ~180.16 g/mol, which match the textbook values.

In [3]:
SYMBOL_INDEX = 0
QUANTITY_INDEX = 1


def compute_molar_mass(symbol_quantity_list, periodic_table_dict):
    """Sum atomic_mass * quantity over every [symbol, quantity] pair.

    For [["H", 2], ["O", 1]]: 1.00794 * 2 + 15.9994 * 1 = 18.01528
    """
    total_molar_mass = 0.0
    for symbol, quantity in symbol_quantity_list:
        atomic_mass = periodic_table_dict[symbol][ATOMIC_MASS_INDEX]
        total_molar_mass += atomic_mass * quantity
    return total_molar_mass


for formula in ["H2O", "C6H12O6", "NaCl", "C2H5OH"]:
    compound = parse_formula(formula, periodic_table)
    mass = compute_molar_mass(compound, periodic_table)
    print(f"{formula:<10} molar mass = {mass:8.3f} g/mol   from {compound}")

H2O        molar mass =   18.015 g/mol   from [('H', 2), ('O', 1)]
C6H12O6    molar mass =  180.156 g/mol   from [('C', 6), ('H', 12), ('O', 6)]
NaCl       molar mass =   58.443 g/mol   from [('Na', 1), ('Cl', 1)]
C2H5OH     molar mass =   46.068 g/mol   from [('C', 2), ('H', 6), ('O', 1)]


## 4. Watching the exception fire

The point of a custom exception is that bad input fails *loudly and specifically*. Here `Xz3`
(unknown element), `H2O)` (unmatched parenthesis), and `C0H2` (zero quantity) each raise
`FormulaError`, while the valid `H2O` passes through.

In [4]:
# The parser raises FormulaError on a bad formula. Because FormulaError subclasses
# ValueError, either except clause below would catch it; we catch the specific type.
bad_formulas = ["H2O", "Xz3", "H2O)", "C0H2"]

for formula in bad_formulas:
    try:
        compound = parse_formula(formula, periodic_table)
        print(f"{formula:<6} OK   -> {compound}")
    except FormulaError as err:
        print(f"{formula:<6} FAIL -> FormulaError: {err.args[0]}")

H2O    OK   -> [('H', 2), ('O', 1)]
Xz3    FAIL -> FormulaError: invalid formula; unknown element symbol: X
H2O)   FAIL -> FormulaError: invalid formula; unmatched close parenthesis
C0H2   FAIL -> FormulaError: invalid formula, quantity begins with zero (0); perhaps you meant capital O for Oxygen


## 5. Inline checks

A few `assert` statements in the spirit of the [testing page](../testing-with-pytest.md) —
known molecules with known masses, the parenthesized-group count, and a confirmation that the
custom exception fires. If any line were wrong, the notebook would stop here.

In [5]:
# Inline pytest-style checks: known molecules with known molar masses.
import math

def approx(a, b, tol=1e-3):
    return math.isclose(a, b, abs_tol=tol)

# Water: 1.00794*2 + 15.9994 = 18.01528
assert compute_molar_mass(parse_formula("H2O", periodic_table), periodic_table) \
    == periodic_table["H"][ATOMIC_MASS_INDEX] * 2 + periodic_table["O"][ATOMIC_MASS_INDEX]

# Glucose C6H12O6 ~ 180.156 g/mol
assert approx(compute_molar_mass(parse_formula("C6H12O6", periodic_table), periodic_table),
              180.156, tol=0.01)

# Parenthesized-group counting: (CH2)12 contributes 12 C and 24 H.
compound = dict(parse_formula("PO4H2(CH2)12CH3", periodic_table))
assert compound["C"] == 13 and compound["H"] == 29 and compound["O"] == 4 and compound["P"] == 1

# The custom exception fires on an unknown symbol.
try:
    parse_formula("Xz9", periodic_table)
    raised = False
except FormulaError:
    raised = True
assert raised, "expected FormulaError for an unknown element symbol"

print("All assertions passed.")

All assertions passed.
